> 在线讲义：[第 5 章 · 推理引擎：从 PyTorch 到 TensorRT，再到 CUDA Graph](https://wikieden.github.io/robot-edge-inference/lessons/05-inference-engines.html) · 本书《AI Infra 扩展 · 机器人端侧推理》

# 第 5 章 · 推理引擎：从 PyTorch 到 TensorRT，再到 CUDA Graph

配套讲义：仓库里 `ai-infra-edge/notes/lessons/05-inference-engines.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import math
import random

print("纯 Python，不需要安装任何库")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（Thor 的数来自 GR00T 2 白皮书 Fig. 19；5 µs、2000 个 kernel 是编的）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第三步 每步省了多少 | 上一行 − 这一行 | `savings(THOR_WATERFALL)` | TensorRT 省 358.41 ms，占 49.8% |
| 第三步 总加速 | 基线 ÷ 最终 | `720.41 / 123.97` | 5.81 倍 |
| 第一步 提交开销 | $2000 \times 5$ µs | `host_launch_ms(2000, 5)` | 10 ms |

Jetson Thor 的逐项耗时来自 GR00T 2 白皮书 Fig. 19；每次 launch 5 µs、一步 2000 个 kernel 是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
THOR_WATERFALL = [  # (累计改动, ms)，Jetson Thor 120 W，G1 Turbo，500 次中位数
    ("基线 eager BF16 PyTorch", 720.41),
    ("+ 运行时优化", 614.53),
    ("+ TensorRT", 256.12),
    ("+ CUDA graph", 254.00),
    ("+ FP8 混合精度", 182.22),
    ("+ 去噪步数减半", 123.97),
]

**函数 `savings`**：每一步比上一步省了多少 ms，占基线的几成。

In [ ]:
def savings(waterfall: list[tuple[str, float]]) -> list[tuple[str, float, float]]:
    """每一步比上一步省了多少 ms，占基线的几成。"""
    base = waterfall[0][1]
    out = []
    for (_, prev), (name, cur) in zip(waterfall, waterfall[1:]):
        out.append((name, prev - cur, (prev - cur) / base))
    return out

**函数 `host_launch_ms`**：CPU 一个个提交 kernel 要多久：kernel 数 × 每次 launch 的时间。

In [ ]:
def host_launch_ms(n_kernels: int, us_per_launch: float) -> float:
    """CPU 一个个提交 kernel 要多久：kernel 数 × 每次 launch 的时间。"""
    return n_kernels * us_per_launch / 1000

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步：GR00T 2 在 Jetson Thor 上一层层削延迟（Fig. 19）")
base = THOR_WATERFALL[0][1]
for name, saved, frac in savings(THOR_WATERFALL):
    print(f"  {name:<16} 省 {saved:7.2f} ms（占基线 {frac:5.1%}）")
print(f"  合计 {base} → {THOR_WATERFALL[-1][1]} ms，快 {base / THOR_WATERFALL[-1][1]:.2f} 倍")

In [ ]:
print("\n第一步：kernel 一个个提交，CPU 光提交就要多久（5 µs、2000 个是编的）")
eager = host_launch_ms(2000, 5)
print(f"  2000 个 kernel × 5 µs = {eager:.0f} ms")
print("  录成 CUDA Graph 后每步只提交 1 次 → 约 0.005 ms；kernel 本身还是要跑")

## 3. 代码里才看得到的

- 瀑布是**累计**的：每一行都包含前面所有优化。如果换一个顺序做（先 FP8 再 TensorRT），每一项的收益会变，所以"TensorRT 省了一半"是在这个顺序下的说法。
- 白皮书给的是 500 次的中位数；控制回路更关心 p99（第 6 章），中位数快不代表最坏情况也快。

## 4. 自己改一改

- 把 `THOR_WATERFALL` 换成 gr00t2 第 10 章里 RTX 5090 那一列（224.73 → 38.40），看 TensorRT 那一步占比是不是也最大。
- 把 `host_launch_ms` 的每次 launch 改成 2 µs，看 2000 个 kernel 还要多久。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一件一结账太慢，整单提交 GPU 不等；
> 合并挑核定精度，引擎认机不认人；
> TensorRT 砍一半，Graph 收尾剩不多；
> 量化少步还要验，形状固定才能录。